# Model Experiments — MedRisk Readmission Prediction

This notebook trains and compares four classification models for 30-day hospital readmission prediction, evaluates them with cross-validation, and uses SHAP to explain the best model's predictions.

In [ ]:
%matplotlib inline

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, f1_score, precision_score, recall_score,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
CV_FOLDS = 5

## 1. Setup — Load, Engineer, Preprocess, Split

In [ ]:
from src.features.engineer import run_feature_engineering
from src.data.preprocessor import clean_data, build_preprocessing_pipeline, split_data

try:
    from src.data.loader import load_raw_data, deduplicate_patients
    df = load_raw_data()
    df = deduplicate_patients(df)
    print("Loaded full dataset.")
except Exception:
    df = pd.read_csv("../data/sample/sample_data.csv", na_values=["?"])
    print("Full dataset not available — using sample data.")

df = run_feature_engineering(df)
df = clean_data(df)

train_df, val_df, test_df = split_data(df, target_col="readmitted", test_size=0.15, val_size=0.15, random_state=RANDOM_STATE)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Target rate — Train: {train_df['readmitted'].mean():.3f}, Test: {test_df['readmitted'].mean():.3f}")

In [ ]:
numeric_features = [
    "age_ordinal", "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_diagnoses", "num_med_changes", "num_meds_active",
    "total_prior_visits", "insulin_changed", "high_utilizer", "medical_specialty_missing",
]
categorical_features = ["race", "gender", "diag1_category", "diag2_category", "diag3_category", "admission_type_id"]

num_avail = [c for c in numeric_features if c in df.columns]
cat_avail = [c for c in categorical_features if c in df.columns]
all_features = num_avail + cat_avail

preprocessor = build_preprocessing_pipeline(num_avail, cat_avail)

X_train = preprocessor.fit_transform(train_df[all_features])
X_val = preprocessor.transform(val_df[all_features])
X_test = preprocessor.transform(test_df[all_features])

y_train = train_df["readmitted"].values
y_val = val_df["readmitted"].values
y_test = test_df["readmitted"].values

X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

feature_names = num_avail + list(
    preprocessor.named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(cat_avail)
)

print(f"Feature matrix shape: {X_train_full.shape}")
print(f"Positive class rate: {y_train_full.mean():.3f}")

## 2. Model Training

In [ ]:
pos_weight = (y_train_full == 0).sum() / max((y_train_full == 1).sum(), 1)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", random_state=RANDOM_STATE, max_iter=1000,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        scale_pos_weight=pos_weight, eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        scale_pos_weight=pos_weight, verbose=-1,
        random_state=RANDOM_STATE,
    ),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_full, y_train_full)
    trained_models[name] = model
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"{name:25s} | ROC-AUC: {roc_auc_score(y_test, y_prob):.4f} | PR-AUC: {average_precision_score(y_test, y_prob):.4f}")

## 3. Cross-Validation Results

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_results = []
for name, model in models.items():
    fold_roc = []
    fold_pr = []
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train_full, y_train_full)):
        X_f_train, X_f_val = X_train_full[train_idx], X_train_full[val_idx]
        y_f_train, y_f_val = y_train_full[train_idx], y_train_full[val_idx]
        model.fit(X_f_train, y_f_train)
        y_prob = model.predict_proba(X_f_val)[:, 1]
        fold_roc.append(roc_auc_score(y_f_val, y_prob))
        fold_pr.append(average_precision_score(y_f_val, y_prob))
    for i in range(CV_FOLDS):
        cv_results.append({"Model": name, "Fold": i + 1, "ROC-AUC": fold_roc[i], "PR-AUC": fold_pr[i]})

cv_df = pd.DataFrame(cv_results)
summary = cv_df.groupby("Model")[["ROC-AUC", "PR-AUC"]].agg(["mean", "std"]).round(4)
summary.columns = [f"{metric} {stat}" for metric, stat in summary.columns]
summary = summary.sort_values("PR-AUC mean", ascending=False)
summary

In [ ]:
print("Per-Fold Results:")
cv_df.pivot_table(index="Fold", columns="Model", values=["ROC-AUC", "PR-AUC"]).round(4)

## 4. ROC Curves

In [ ]:
# Refit all models on full train set for test evaluation
for name, model in models.items():
    model.fit(X_train_full, y_train_full)
    trained_models[name] = model

fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6"]

for (name, model), color in zip(trained_models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5. Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
baseline = y_test.mean()

for (name, model), color in zip(trained_models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(recall, precision, color=color, linewidth=2, label=f"{name} (AP={ap:.3f})")

ax.axhline(baseline, color="gray", linestyle="--", linewidth=1, label=f"Baseline ({baseline:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — All Models")
ax.legend(loc="upper right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

## 6. Threshold Analysis

Precision/recall trade-off across thresholds for the best model (by PR-AUC on test).

In [ ]:
# Identify the best model by PR-AUC on test set
test_pr_aucs = {}
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    test_pr_aucs[name] = average_precision_score(y_test, y_prob)

best_model_name = max(test_pr_aucs, key=test_pr_aucs.get)
best_model = trained_models[best_model_name]
print(f"Best model: {best_model_name} (PR-AUC = {test_pr_aucs[best_model_name]:.4f})")

y_prob_best = best_model.predict_proba(X_test)[:, 1]
precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob_best)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precision_vals[:-1], "b-", linewidth=2, label="Precision")
ax.plot(thresholds, recall_vals[:-1], "r-", linewidth=2, label="Recall")

# Mark 70% recall target
target_recall = 0.70
valid = recall_vals[:-1] >= target_recall
if valid.any():
    opt_idx = np.where(valid)[0][-1]
    opt_threshold = thresholds[opt_idx]
    ax.axvline(opt_threshold, color="green", linestyle="--", linewidth=1.5,
               label=f"Optimal threshold ({opt_threshold:.3f})")
    print(f"Optimal threshold for ≥70% recall: {opt_threshold:.3f}")
    print(f"  Precision at threshold: {precision_vals[opt_idx]:.3f}")
    print(f"  Recall at threshold: {recall_vals[opt_idx]:.3f}")

ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis — {best_model_name}")
ax.legend()
plt.tight_layout()
plt.show()

## 7. SHAP Global Summary

Beeswarm plot showing how each feature contributes to predictions across the test set.

In [ ]:
import shap

# Use a subsample for speed if test set is large
X_shap = X_test[:min(500, len(X_test))]

if hasattr(best_model, "get_booster") or isinstance(best_model, (XGBClassifier, LGBMClassifier, RandomForestClassifier)):
    explainer = shap.TreeExplainer(best_model)
else:
    explainer = shap.LinearExplainer(best_model, X_train_full)

shap_values = explainer.shap_values(X_shap)

# For binary classifiers that return a list, take the positive class
if isinstance(shap_values, list):
    shap_values = shap_values[1]

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False, max_display=20)
plt.title(f"SHAP Summary — {best_model_name}", fontsize=13)
plt.tight_layout()
plt.show()

## 8. SHAP Feature Importance Bar

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
shap_importance.head(20).sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title(f"Mean |SHAP| Feature Importance — {best_model_name}")
ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.show()

## 9. Model Comparison Summary

In [ ]:
comparison = []
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    comparison.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
    })

comparison_df = pd.DataFrame(comparison).set_index("Model").sort_values("PR-AUC", ascending=False)
comparison_df = comparison_df.round(4)

print("Final Model Comparison (Test Set):")
comparison_df.style.highlight_max(axis=0, color="lightgreen")